# Kabyle-XLS-R zero-shot and adapted evaluation on Tarifit

This notebook performs three frozen evaluations:

1. Native `Akashpb13/Kabyle_xlsr` zero-shot on the controlled V1.2 validation set.
2. The same native checkpoint zero-shot on the frozen V1.3 final test.
3. The selected epoch-8 Kabyle-XLS-R → Tarifit model on the same V1.3 final test.

No training or checkpoint selection occurs here. V1.3 is used once for final scoring and must not guide further tuning. Greedy CTC decoding is used throughout, special tokens are removed, and no Kabyle-to-Tarifit orthographic mapping or language model is applied.


In [ ]:
# Cell 1 — Install the fixed evaluation dependencies

!pip -q install "transformers==4.57.1" "datasets==4.4.1" "accelerate>=1.10,<2" "jiwer==4.0.0" "safetensors>=0.4.5" "soundfile>=0.12.1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 152.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 122.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 130.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
# Cell 2 — Mount Drive and define the frozen evaluation paths

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

SAFE_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/tarifit_asr_tfm"
)

V1_2_FROZEN_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "segments_metadata_v1_2_train_val_frozen.csv"
)

V1_3_METADATA_ROOT = PROJECT_ROOT / "data" / "metadata"

ADAPTED_MODEL_DIR = (
    SAFE_PROJECT_ROOT
    / "models"
    / "kabyle_xlsr_tarifit_v1_2_noaug_full8_corrected"
    / "best_model"
)

RESULTS_DIR = (
    SAFE_PROJECT_ROOT
    / "results"
    / "kabyle_xlsr_zero_shot_and_adapted_final_evaluation"
)

ZERO_SHOT_MODEL_ID = "Akashpb13/Kabyle_xlsr"
RUN_FINAL_V1_3_TEST = True

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert V1_2_FROZEN_PATH.exists(), V1_2_FROZEN_PATH
assert V1_3_METADATA_ROOT.exists(), V1_3_METADATA_ROOT
assert RUN_FINAL_V1_3_TEST is True

print("Zero-shot checkpoint:", ZERO_SHOT_MODEL_ID)
print("Adapted checkpoint:", ADAPTED_MODEL_DIR)
print("V1.2 metadata:", V1_2_FROZEN_PATH)
print("V1.3 metadata root:", V1_3_METADATA_ROOT)
print("Results:", RESULTS_DIR)
print("Final-test configurations frozen: yes")


Mounted at /content/drive
Zero-shot checkpoint: Akashpb13/Kabyle_xlsr
Adapted checkpoint: /content/drive/MyDrive/tarifit_asr_tfm/models/kabyle_xlsr_tarifit_v1_2_noaug_full8_corrected/best_model
V1.2 metadata: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/metadata/segments_metadata_v1_2_train_val_frozen.csv
V1.3 metadata root: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/metadata
Results: /content/drive/MyDrive/tarifit_asr_tfm/results/kabyle_xlsr_zero_shot_and_adapted_final_evaluation
Final-test configurations frozen: yes


In [ ]:
# Cell 3 — Import evaluation libraries and select the device

import gc
import hashlib
import json
import re
import unicodedata

import numpy as np
import pandas as pd
import soundfile as sf
import torch

from jiwer import wer, cer
from transformers import AutoModelForCTC, AutoProcessor

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: NVIDIA L4


In [ ]:
# Cell 4 — Define shared metadata and text-normalization rules

def evaluation_normalize(text):
    text = unicodedata.normalize("NFC", str(text))
    text = text.lower().strip()
    return re.sub(r"\s+", " ", text)


def standardize_metadata(frame, required_split=None):
    frame = frame.copy()

    if "transcription" not in frame.columns:
        if "transcript_normalized" in frame.columns:
            frame["transcription"] = frame["transcript_normalized"]
        else:
            raise ValueError(
                "Expected transcription or transcript_normalized."
            )

    if required_split is not None and "dataset_split" in frame.columns:
        split_values = (
            frame["dataset_split"]
            .astype(str)
            .str.lower()
            .str.strip()
        )
        frame = frame.loc[split_values == required_split].copy()

    if "segment_id" not in frame.columns:
        raise ValueError("Expected a segment_id column.")

    if "audio_path" not in frame.columns:
        if "absolute_audio_path" not in frame.columns:
            raise ValueError(
                "Expected audio_path or absolute_audio_path."
            )
        frame["audio_path"] = frame["absolute_audio_path"]

    frame["transcription"] = frame["transcription"].map(
        evaluation_normalize
    )

    assert frame["transcription"].ne("").all()
    assert not frame["segment_id"].duplicated().any()

    return frame.reset_index(drop=True)


def resolve_audio_path(raw_path):
    raw_path = Path(str(raw_path))

    candidates = []
    if raw_path.is_absolute():
        candidates.append(raw_path)
    else:
        candidates.append(PROJECT_ROOT / raw_path)

    parts = list(raw_path.parts)
    if "tarifit_asr_tfm" in parts:
        project_index = parts.index("tarifit_asr_tfm")
        relative_tail = Path(*parts[project_index + 1:])
        candidates.append(PROJECT_ROOT / relative_tail)

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        f"Could not resolve audio path: {raw_path}"
    )


def metadata_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


print("Shared evaluation normalization ready.")


Shared evaluation normalization ready.


In [ ]:
# Cell 5 — Load and verify the controlled V1.2 validation set

v1_2_frozen = pd.read_csv(V1_2_FROZEN_PATH)
v1_2_validation = standardize_metadata(
    v1_2_frozen,
    required_split="validation",
)

assert len(v1_2_validation) == 129, len(v1_2_validation)

if "duration_seconds" in v1_2_validation.columns:
    v1_2_seconds = float(
        v1_2_validation["duration_seconds"].sum()
    )
    assert abs(v1_2_seconds - 1074.088) < 1.0, v1_2_seconds
else:
    v1_2_seconds = None

missing_v1_2_audio = []
for path in v1_2_validation["audio_path"]:
    try:
        resolve_audio_path(path)
    except FileNotFoundError:
        missing_v1_2_audio.append(path)

assert not missing_v1_2_audio, missing_v1_2_audio[:3]

print("V1.2 validation segments:", len(v1_2_validation))
print("V1.2 validation seconds:", v1_2_seconds)
print("V1.2 metadata SHA-256:", metadata_sha256(V1_2_FROZEN_PATH))
print("Missing V1.2 audio:", len(missing_v1_2_audio))


V1.2 validation segments: 129
V1.2 validation seconds: 1074.088
V1.2 metadata SHA-256: 4911f46e0a8c3656089677b8899d8296b9e1528d8aa3633a64728eb645f28fab
Missing V1.2 audio: 0


In [ ]:
# Cell 6 — Locate and verify the frozen V1.3 final-test metadata

def v1_3_candidate_paths():
    paths = []
    for path in V1_3_METADATA_ROOT.rglob("*.csv"):
        lowered = str(path).lower()
        if "v1_3" in lowered or "v1.3" in lowered:
            paths.append(path)
    return sorted(set(paths))


valid_v1_3_candidates = []
candidate_diagnostics = []

for candidate_path in v1_3_candidate_paths():
    try:
        candidate_raw = pd.read_csv(candidate_path)
        candidate_frame = standardize_metadata(
            candidate_raw,
            required_split="test",
        )

        candidate_seconds = None
        if "duration_seconds" in candidate_frame.columns:
            candidate_seconds = float(
                candidate_frame["duration_seconds"].sum()
            )

        candidate_diagnostics.append(
            (
                str(candidate_path),
                len(candidate_frame),
                candidate_seconds,
            )
        )

        count_matches = len(candidate_frame) == 115
        duration_matches = (
            candidate_seconds is None
            or abs(candidate_seconds - 1075.264) < 1.0
        )

        if count_matches and duration_matches:
            valid_v1_3_candidates.append(
                (candidate_path, candidate_frame, candidate_seconds)
            )
    except Exception as error:
        candidate_diagnostics.append(
            (str(candidate_path), "rejected", repr(error))
        )

print("V1.3 candidate diagnostics:")
for diagnostic in candidate_diagnostics:
    print(diagnostic)

if not valid_v1_3_candidates:
    raise FileNotFoundError(
        "No V1.3 CSV matched the frozen test invariants: "
        "115 segments and approximately 1075.264 seconds."
    )

candidate_signatures = []
for _, frame, _ in valid_v1_3_candidates:
    signature = tuple(
        sorted(
            zip(
                frame["segment_id"].astype(str),
                frame["transcription"].astype(str),
            )
        )
    )
    candidate_signatures.append(signature)

assert all(
    signature == candidate_signatures[0]
    for signature in candidate_signatures
), "Multiple conflicting V1.3 test manifests were found."

V1_3_TEST_PATH, v1_3_test, v1_3_seconds = (
    valid_v1_3_candidates[0]
)

missing_v1_3_audio = []
for path in v1_3_test["audio_path"]:
    try:
        resolve_audio_path(path)
    except FileNotFoundError:
        missing_v1_3_audio.append(path)

assert not missing_v1_3_audio, missing_v1_3_audio[:3]

print("Selected V1.3 manifest:", V1_3_TEST_PATH)
print("V1.3 final-test segments:", len(v1_3_test))
print("V1.3 final-test seconds:", v1_3_seconds)
print("V1.3 metadata SHA-256:", metadata_sha256(V1_3_TEST_PATH))
print("Missing V1.3 audio:", len(missing_v1_3_audio))


V1.3 candidate diagnostics:
('/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/metadata/final_test_v1_3.csv', 115, 1075.264)
('/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/metadata/final_test_v1_3_pre_consistency_review.csv', 115, 1075.264)
('/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/metadata/final_test_v1_3_pre_normalization.csv', 115, 1075.264)
('/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/metadata/segments_metadata_v1_3.csv', 115, 1075.264)
('/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/metadata/segments_metadata_v1_3_final_eval.csv', 115, 1075.264)


AssertionError: Multiple conflicting V1.3 test manifests were found.

In [ ]:
# Cell 6 — Load and verify the canonical frozen V1.3 final-test metadata

V1_3_TEST_PATH = (
    V1_3_METADATA_ROOT
    / "segments_metadata_v1_3_final_eval.csv"
)

assert V1_3_TEST_PATH.exists(), V1_3_TEST_PATH

v1_3_raw = pd.read_csv(V1_3_TEST_PATH)

v1_3_test = standardize_metadata(
    v1_3_raw,
    required_split="test",
)

assert len(v1_3_test) == 115, len(v1_3_test)
assert not v1_3_test["segment_id"].duplicated().any()

assert "duration_seconds" in v1_3_test.columns
v1_3_seconds = float(v1_3_test["duration_seconds"].sum())

assert abs(v1_3_seconds - 1075.264) < 1.0, v1_3_seconds

if "speaker_group_id" in v1_3_test.columns:
    speaker_ids = sorted(
        v1_3_test["speaker_group_id"].dropna().unique()
    )
    print("Held-out speakers:", speaker_ids)

missing_v1_3_audio = []

for raw_path in v1_3_test["audio_path"]:
    try:
        resolve_audio_path(raw_path)
    except FileNotFoundError:
        missing_v1_3_audio.append(raw_path)

assert not missing_v1_3_audio, missing_v1_3_audio[:3]

print("Selected V1.3 manifest:", V1_3_TEST_PATH)
print("V1.3 final-test segments:", len(v1_3_test))
print("V1.3 final-test seconds:", v1_3_seconds)
print("V1.3 metadata SHA-256:", metadata_sha256(V1_3_TEST_PATH))
print("Missing V1.3 audio:", len(missing_v1_3_audio))

Held-out speakers: ['SPK003', 'SPK004']
Selected V1.3 manifest: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/metadata/segments_metadata_v1_3_final_eval.csv
V1.3 final-test segments: 115
V1.3 final-test seconds: 1075.264
V1.3 metadata SHA-256: fd496632b179c97ab18189d0300b0fd84d1cda1d17a8071d4b3dc108b2ca1f70
Missing V1.3 audio: 0


In [ ]:
# Cell 7 — Define one shared greedy-CTC evaluation function

def evaluate_system(
    frame,
    processor,
    model,
    system_name,
    partition_name,
    output_path,
    batch_size=2,
):
    model = model.to(DEVICE)
    model.eval()

    predictions = []
    references = []
    segment_ids = []

    for start in range(0, len(frame), batch_size):
        batch_frame = frame.iloc[start:start + batch_size]
        audio_arrays = []

        for row in batch_frame.itertuples(index=False):
            audio_path = resolve_audio_path(row.audio_path)
            audio, sampling_rate = sf.read(
                audio_path,
                dtype="float32",
                always_2d=False,
            )

            if sampling_rate != 16000:
                raise ValueError(
                    f"{row.segment_id}: expected 16000 Hz, "
                    f"found {sampling_rate} Hz"
                )

            if getattr(audio, "ndim", 1) != 1:
                raise ValueError(
                    f"{row.segment_id}: expected mono audio"
                )

            audio_arrays.append(audio)

        inputs = processor(
            audio_arrays,
            sampling_rate=16000,
            padding=True,
            return_tensors="pt",
        )
        inputs = {
            key: value.to(DEVICE)
            for key, value in inputs.items()
        }

        with torch.inference_mode():
            logits = model(**inputs).logits

        prediction_ids = torch.argmax(
            logits,
            dim=-1,
        ).cpu()

        batch_predictions = processor.batch_decode(
            prediction_ids,
            skip_special_tokens=True,
        )

        predictions.extend(
            evaluation_normalize(text)
            for text in batch_predictions
        )
        references.extend(
            evaluation_normalize(text)
            for text in batch_frame["transcription"]
        )
        segment_ids.extend(
            batch_frame["segment_id"].astype(str).tolist()
        )

        if start % 20 == 0:
            print(
                f"{system_name} / {partition_name}: "
                f"{min(start + batch_size, len(frame))}/{len(frame)}"
            )

    assert len(predictions) == len(references) == len(frame)
    assert not any("[UNK]" in text for text in predictions)

    result_frame = pd.DataFrame(
        {
            "segment_id": segment_ids,
            "reference": references,
            "prediction": predictions,
        }
    )

    result_frame["segment_wer"] = [
        wer(reference, prediction)
        for reference, prediction in zip(
            references,
            predictions,
        )
    ]
    result_frame["segment_cer"] = [
        cer(reference, prediction)
        for reference, prediction in zip(
            references,
            predictions,
        )
    ]

    metrics = {
        "system": system_name,
        "partition": partition_name,
        "segments": len(frame),
        "wer_percent": float(wer(references, predictions) * 100),
        "cer_percent": float(cer(references, predictions) * 100),
        "empty_hypotheses": int(
            sum(not prediction for prediction in predictions)
        ),
        "decoding": "greedy CTC",
        "language_model": False,
        "orthographic_mapping": False,
        "special_tokens_removed": True,
    }

    result_frame.to_csv(
        output_path,
        index=False,
        encoding="utf-8",
    )

    print(json.dumps(metrics, ensure_ascii=False, indent=2))
    print("Saved predictions:", output_path)
    return metrics, result_frame


print("Shared greedy-CTC evaluator ready.")


Shared greedy-CTC evaluator ready.


In [ ]:
# Cell 8 — Load the native Kabyle-XLS-R checkpoint

zero_shot_processor = AutoProcessor.from_pretrained(
    ZERO_SHOT_MODEL_ID
)

zero_shot_model = AutoModelForCTC.from_pretrained(
    ZERO_SHOT_MODEL_ID
)

print("Loaded:", ZERO_SHOT_MODEL_ID)
print("Native vocabulary size:", zero_shot_model.config.vocab_size)
print(
    "Native padding/blank id:",
    zero_shot_model.config.pad_token_id,
)
print(
    "Tokenizer padding id:",
    zero_shot_processor.tokenizer.pad_token_id,
)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/221 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loaded: Akashpb13/Kabyle_xlsr
Native vocabulary size: 57
Native padding/blank id: 56
Tokenizer padding id: 56


In [ ]:
# Cell 9 — Evaluate native Kabyle-XLS-R zero-shot on V1.2

V1_2_ZERO_SHOT_PREDICTIONS = (
    RESULTS_DIR
    / "kabyle_xlsr_zero_shot_v1_2_validation_predictions.csv"
)

v1_2_zero_shot_metrics, v1_2_zero_shot_predictions = (
    evaluate_system(
        frame=v1_2_validation,
        processor=zero_shot_processor,
        model=zero_shot_model,
        system_name="Kabyle-XLS-R zero-shot",
        partition_name="V1.2 validation",
        output_path=V1_2_ZERO_SHOT_PREDICTIONS,
    )
)

print("V1.2 zero-shot evaluation complete.")


Kabyle-XLS-R zero-shot / V1.2 validation: 2/129
Kabyle-XLS-R zero-shot / V1.2 validation: 22/129
Kabyle-XLS-R zero-shot / V1.2 validation: 42/129
Kabyle-XLS-R zero-shot / V1.2 validation: 62/129
Kabyle-XLS-R zero-shot / V1.2 validation: 82/129
Kabyle-XLS-R zero-shot / V1.2 validation: 102/129
Kabyle-XLS-R zero-shot / V1.2 validation: 122/129
{
  "system": "Kabyle-XLS-R zero-shot",
  "partition": "V1.2 validation",
  "segments": 129,
  "wer_percent": 103.6139455782313,
  "cer_percent": 56.194227831819276,
  "empty_hypotheses": 0,
  "decoding": "greedy CTC",
  "language_model": false,
  "orthographic_mapping": false,
  "special_tokens_removed": true
}
Saved predictions: /content/drive/MyDrive/tarifit_asr_tfm/results/kabyle_xlsr_zero_shot_and_adapted_final_evaluation/kabyle_xlsr_zero_shot_v1_2_validation_predictions.csv
V1.2 zero-shot evaluation complete.


In [ ]:
# Cell 10 — Evaluate native Kabyle-XLS-R once on V1.3

assert RUN_FINAL_V1_3_TEST is True

V1_3_ZERO_SHOT_PREDICTIONS = (
    RESULTS_DIR
    / "kabyle_xlsr_zero_shot_v1_3_test_predictions.csv"
)

v1_3_zero_shot_metrics, v1_3_zero_shot_predictions = (
    evaluate_system(
        frame=v1_3_test,
        processor=zero_shot_processor,
        model=zero_shot_model,
        system_name="Kabyle-XLS-R zero-shot",
        partition_name="V1.3 final test",
        output_path=V1_3_ZERO_SHOT_PREDICTIONS,
    )
)

del zero_shot_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("V1.3 zero-shot evaluation complete.")


Kabyle-XLS-R zero-shot / V1.3 final test: 2/115
Kabyle-XLS-R zero-shot / V1.3 final test: 22/115
Kabyle-XLS-R zero-shot / V1.3 final test: 42/115
Kabyle-XLS-R zero-shot / V1.3 final test: 62/115
Kabyle-XLS-R zero-shot / V1.3 final test: 82/115
Kabyle-XLS-R zero-shot / V1.3 final test: 102/115
{
  "system": "Kabyle-XLS-R zero-shot",
  "partition": "V1.3 final test",
  "segments": 115,
  "wer_percent": 90.43650793650794,
  "cer_percent": 35.44456252213956,
  "empty_hypotheses": 0,
  "decoding": "greedy CTC",
  "language_model": false,
  "orthographic_mapping": false,
  "special_tokens_removed": true
}
Saved predictions: /content/drive/MyDrive/tarifit_asr_tfm/results/kabyle_xlsr_zero_shot_and_adapted_final_evaluation/kabyle_xlsr_zero_shot_v1_3_test_predictions.csv
V1.3 zero-shot evaluation complete.


In [ ]:
# Cell 11 — Load and verify the selected adapted Tarifit model

def model_weight_files(directory):
    directory = Path(directory)
    candidates = [
        directory / "model.safetensors",
        directory / "pytorch_model.bin",
        directory / "model.safetensors.index.json",
        directory / "pytorch_model.bin.index.json",
    ]
    candidates.extend(directory.glob("model-*.safetensors"))
    candidates.extend(directory.glob("pytorch_model-*.bin"))
    return sorted({path for path in candidates if path.exists()})


adapted_weight_files = model_weight_files(ADAPTED_MODEL_DIR)
assert adapted_weight_files, (
    f"No adapted-model weights found in {ADAPTED_MODEL_DIR}. "
    "Run Cells 18–20 of the training notebook first."
)

adapted_processor = AutoProcessor.from_pretrained(
    str(ADAPTED_MODEL_DIR)
)

adapted_model = AutoModelForCTC.from_pretrained(
    str(ADAPTED_MODEL_DIR)
)

assert adapted_model.config.vocab_size == 34
assert len(adapted_processor.tokenizer) == 34

print("Loaded adapted model:", ADAPTED_MODEL_DIR)
print(
    "Verified adapted weights:",
    [path.name for path in adapted_weight_files],
)
print("Adapted Tarifit vocabulary size:", adapted_model.config.vocab_size)


Loaded adapted model: /content/drive/MyDrive/tarifit_asr_tfm/models/kabyle_xlsr_tarifit_v1_2_noaug_full8_corrected/best_model
Verified adapted weights: ['model.safetensors']
Adapted Tarifit vocabulary size: 34


In [ ]:
# Cell 12 — Evaluate the selected adapted model once on V1.3

assert RUN_FINAL_V1_3_TEST is True

V1_3_ADAPTED_PREDICTIONS = (
    RESULTS_DIR
    / "kabyle_xlsr_adapted_v1_3_test_predictions.csv"
)

v1_3_adapted_metrics, v1_3_adapted_predictions = (
    evaluate_system(
        frame=v1_3_test,
        processor=adapted_processor,
        model=adapted_model,
        system_name="Kabyle-XLS-R → Tarifit",
        partition_name="V1.3 final test",
        output_path=V1_3_ADAPTED_PREDICTIONS,
    )
)

print("V1.3 adapted-model evaluation complete.")


Kabyle-XLS-R → Tarifit / V1.3 final test: 2/115
Kabyle-XLS-R → Tarifit / V1.3 final test: 22/115
Kabyle-XLS-R → Tarifit / V1.3 final test: 42/115
Kabyle-XLS-R → Tarifit / V1.3 final test: 62/115
Kabyle-XLS-R → Tarifit / V1.3 final test: 82/115
Kabyle-XLS-R → Tarifit / V1.3 final test: 102/115
{
  "system": "Kabyle-XLS-R → Tarifit",
  "partition": "V1.3 final test",
  "segments": 115,
  "wer_percent": 89.28571428571429,
  "cer_percent": 30.371944739638685,
  "empty_hypotheses": 0,
  "decoding": "greedy CTC",
  "language_model": false,
  "orthographic_mapping": false,
  "special_tokens_removed": true
}
Saved predictions: /content/drive/MyDrive/tarifit_asr_tfm/results/kabyle_xlsr_zero_shot_and_adapted_final_evaluation/kabyle_xlsr_adapted_v1_3_test_predictions.csv
V1.3 adapted-model evaluation complete.


In [ ]:
# Cell 13 — Save the final comparison and reproducibility record

comparison = pd.DataFrame(
    [
        v1_2_zero_shot_metrics,
        v1_3_zero_shot_metrics,
        v1_3_adapted_metrics,
    ]
)

COMPARISON_PATH = RESULTS_DIR / "evaluation_comparison.csv"
comparison.to_csv(
    COMPARISON_PATH,
    index=False,
    encoding="utf-8",
)

evaluation_record = {
    "zero_shot_model": ZERO_SHOT_MODEL_ID,
    "adapted_model": str(ADAPTED_MODEL_DIR),
    "v1_2_manifest": str(V1_2_FROZEN_PATH),
    "v1_2_manifest_sha256": metadata_sha256(V1_2_FROZEN_PATH),
    "v1_3_manifest": str(V1_3_TEST_PATH),
    "v1_3_manifest_sha256": metadata_sha256(V1_3_TEST_PATH),
    "final_test_used_for_tuning": False,
    "decoding": "greedy CTC without a language model",
    "kabyle_to_tarifit_mapping": False,
    "special_tokens_removed": True,
    "results": comparison.to_dict(orient="records"),
    "prediction_files": {
        "v1_2_zero_shot": str(V1_2_ZERO_SHOT_PREDICTIONS),
        "v1_3_zero_shot": str(V1_3_ZERO_SHOT_PREDICTIONS),
        "v1_3_adapted": str(V1_3_ADAPTED_PREDICTIONS),
    },
}

RECORD_PATH = RESULTS_DIR / "evaluation_record.json"
with open(RECORD_PATH, "w", encoding="utf-8") as file:
    json.dump(
        evaluation_record,
        file,
        ensure_ascii=False,
        indent=2,
    )

display(
    comparison[
        [
            "system",
            "partition",
            "segments",
            "wer_percent",
            "cer_percent",
            "empty_hypotheses",
        ]
    ]
)

print("Saved comparison:", COMPARISON_PATH)
print("Saved evaluation record:", RECORD_PATH)
print("No further model tuning may use the V1.3 results.")


,system,partition,segments,wer_percent,cer_percent,empty_hypotheses
0,Kabyle-XLS-R zero-shot,V1.2 validation,129,103.613946,56.194228,0
1,Kabyle-XLS-R zero-shot,V1.3 final test,115,90.436508,35.444563,0
2,Kabyle-XLS-R → Tarifit,V1.3 final test,115,89.285714,30.371945,0


Saved comparison: /content/drive/MyDrive/tarifit_asr_tfm/results/kabyle_xlsr_zero_shot_and_adapted_final_evaluation/evaluation_comparison.csv
Saved evaluation record: /content/drive/MyDrive/tarifit_asr_tfm/results/kabyle_xlsr_zero_shot_and_adapted_final_evaluation/evaluation_record.json
No further model tuning may use the V1.3 results.
